In [ ]:
import jax
import jax.numpy as jnp
from jax.nn import sigmoid, relu
from jax import random
from adaptive_latents.input_sources.autoregressor import AdamOptimizer
import matplotlib.pyplot as plt

key = random.key(0)

In [ ]:
from jax.extend.backend import get_backend
print(get_backend().platform)

## proof sanity check/numerical check

In [ ]:
for _ in range(100):
    key, old_key = jax.random.split(key)
    s = random.uniform(old_key, shape=(130,)) * .1

    key, old_key = jax.random.split(key)
    v = random.normal(old_key, shape=(130,2))

    key, old_key = jax.random.split(key)
    a1, a2 = random.uniform(old_key, shape=(2,))

    steps = []
    steps.append(a1*jnp.linalg.norm(v @ v.T @ s)**2 - a2*jnp.linalg.norm(s - v @ v.T @ s)**2)
    steps.append(a1*s.T @ v @ (v.T @ v) @ v.T @ s - a2*jnp.linalg.norm(s - v @ v.T @ s)**2)
    steps.append(a1*s.T @ v @ (v.T @ v) @ v.T @ s - a2*(s.T @ s - 2 * s.T @ v @ v.T @ s + s.T @ v @ (v.T @ v) @ v.T @ s))
    steps.append(a1*s.T @ v @ (v.T @ v) @ v.T @ s + -a2*s.T @ s - -a2*2 * s.T @ v @ v.T @ s + -a2*s.T @ v @ (v.T @ v) @ v.T @ s)
    steps.append((a1-a2)*s.T @ v @ (v.T @ v) @ v.T @ s + a2*(2 * s.T @ v @ v.T @ s -s.T @ s)  )
    steps.append(s.T @ ((a1-a2)* v @ (v.T @ v) @ v.T + a2*(2 * v @ v.T - jnp.eye(v.shape[0]))) @ s  )


    for i in range(len(steps)-1):
        assert jnp.allclose(steps[i], steps[i+1]), i
    assert jnp.allclose(steps[0], steps[-1])



## using a real Q

In [ ]:
Q = jnp.load('Q.npy')

v = Q[:,:2]
v = jnp.atleast_2d(v.T).T

In [ ]:
def loss(s, v, lam_1=1e-3):
    u = s  # this assumes for now that the dynamics S function is an identity
    return (
        - jnp.sqrt(jnp.linalg.norm(v.T @ s))**2  # maximize dot product with the target vector
        + jnp.linalg.norm(s - v @ v.T @ s)**2  # minimize orthogonal component
        + jnp.linalg.norm(u, ord=1) * lam_1  # L1 penalty
    )

# def loss(s, v, lam_1=1e-3):
#     u = s  # this assumes for now that the dynamics S function is an identity
#     return (
#             - 10 * jnp.linalg.norm(v @ v.T @ s)**2  # maximize dot product with the target vector
#             + jnp.linalg.norm(s - v @ v.T @ s)**2  # minimize orthogonal component
#             + jnp.linalg.norm(u, ord=1) * lam_1  # L1 penalty
#     )
grad_loss = jax.jit(jax.value_and_grad(loss))

In [ ]:
s_history = []
loss_history = []
N = 30

lam_1 = 1e-3
convergence_threshold = 1e-2
while True:
    # make a random s to optimize
    key, old_key = jax.random.split(key)
    s = random.uniform(old_key, shape=(130,)) * .1
    s_optimizer = AdamOptimizer(lr=0.005)

    for i in range(250):
        # Adam update
        val, grad = grad_loss(s, v, lam_1=lam_1)
        s = s_optimizer.update(s,grad)

        # set negative values to 0
        s = relu(s)

        # logging
        s_history.append(s)
        loss_history.append(val)

        # if converged, break
        if len(s_history) > 10 and jnp.linalg.norm(s_history[-2] - s_history[-1]) < convergence_threshold:
            break

    # if the L0 norm is too big, try again with a larger L1 penalty
    l0 = jnp.linalg.norm(s,ord=0)
    if 0 < l0 <= N:
        break
    if l0 == 0 or jnp.isnan(s).any():
        print(f"{jnp.linalg.norm(s, ord=0)}, {lam_1:.2f}, /")
        lam_1 /= 1.2
    else:
        print(f"{jnp.linalg.norm(s, ord=0)}, {lam_1:.2f}, *")
        lam_1 *= 2


In [ ]:
fig, axs = plt.subplots(ncols=2, layout='tight')
axs[0].plot(jnp.linalg.norm(jnp.diff(jnp.array(s_history), axis=0), axis=1))
axs[0].axhline(convergence_threshold, color='r')
axs[0].semilogy()
axs[0].set_xlabel("iteration")
axs[0].set_ylabel("difference between $\Delta s$ norms")
axs[1].plot(loss_history)
axs[1].set_xlabel("iteration")
axs[1].set_ylabel("loss history")



In [ ]:
fig, ax = plt.subplots()
ax.plot(v, color='k', alpha=.1, label='v (the target space)')
ax.plot(s/s.max(), color='C0', label='s (optimized stimulation vector)')
ax.legend()
# ax.plot(s/s.max(), color='C0')
print(jnp.linalg.norm(s, ord=0))

In [ ]:
energies = (Q.T @ s)**2
energies = energies / energies.sum()
fig, ax = plt.subplots()
ax.plot(energies, '.-')
ax.set_xlabel("latent direction")
ax.set_ylabel("sim energy received")



In [ ]:
key, old_key = jax.random.split(key)
X = random.normal(old_key, shape=(10000,20))

key, old_key = jax.random.split(key)
x = random.normal(old_key, shape=(10,))

key, old_key = jax.random.split(key)
y = random.normal(old_key, shape=(10,)) * 0

# X = X.at[100:].set(jnp.nan)

X = X.at[9000:, :10].set(1e100)
X = X.at[9000:, 10:].set(0)


def f(x, y, X):
    w = jnp.exp(-(jnp.linalg.norm(X[:,:10] - x, axis=1))**2)
    w = w / w.sum()

    y_hat = w @ X[:,10:]
    return jnp.linalg.norm(y_hat - y)**2

value_and_grad_f = jax.value_and_grad(f)
value, grad = value_and_grad_f(x,y,X)
grad